In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import swifter
import psutil
import math

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg
%matplotlib inline

import missingno as msno
import pprint

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [2]:
path, dirs, files = next(os.walk("chunks_3/test/"))
directory = "chunks_3/test/"
file_count = len(files)

In [3]:
# count number of rows
from itertools import (takewhile,repeat)
def rawincount(filename):
    f = open(filename, 'rb')
    bufgen = takewhile(lambda x: x, (f.raw.read(1024*1024) for _ in repeat(None)))
    return sum( buf.count(b'\n') for buf in bufgen )

# compute for memory usage
def usage():
    process = psutil.Process(os.getpid())
    return process.memory_info()[0] / float(2 ** 20)

In [4]:
def garbage_collector():
    for i in range(4):
        print('Collecting %d ...' % i)
        n = gc.collect()
        print('Unreachable objects:', n)
        print('Remaining Garbage:',) 
        pprint.pprint(gc.garbage)
        print

In [7]:
files_proc = ['58120', '59409', '59513', '59514', '66987', '77401', '90935', 
              '96408', '99432', '99460', 'MCP01', 'NSD01']

length_list = len(files_proc)

In [32]:
# actual amounts aggregation counts
for i in range(length_list):
    df_input = pd.read_csv(os.path.join(directory + files_proc[i] + '.csv'), low_memory=False)
    df_input = df_input[['YEAR', 'ACTUAL_AMT', 'PAID AMOUNT', 'CHECK_DT', 'RECEIVED/REFILED DATE']]

    # computing for Turnaround Time
    df_input[['CHECK_DT','RECEIVED/REFILED DATE']] = df_input[['CHECK_DT','RECEIVED/REFILED DATE']].apply(pd.to_datetime)
    df_input['TAT'] = (df_input['CHECK_DT'] - df_input['RECEIVED/REFILED DATE']).dt.days

    # computing for Financial Coverage; set x/0 row as NaN so it will not be included in the aggregates
    df_input['FINANCIAL_COVERAGE'] = df_input['PAID AMOUNT']/df_input['ACTUAL_AMT']
    df_input.loc[~np.isfinite(df_input['FINANCIAL_COVERAGE']), 'FINANCIAL_COVERAGE'] = np.nan

    df_input.to_csv(os.path.join(directory, files_proc[i] + '_strip.csv'), index=False)
    print(files_proc[i],len(df_input))
    
    del[df_input]
#     garbage_collector() 
    gc.collect()
    gc.collect()
    gc.collect()

58120 382889
59409 909762
59513 1154665
59514 822261
66987 742169
77401 696447
90935 11988992
96408 925538
99432 4150748
99460 1345360
MCP01 1972131
NSD01 2806238


In [14]:
files_proc = ['90935']
cols_exclude = ['INSTITUTION_NAME', 'INSTITUTION_PROVINCE', 'INSTITUTION_MUNICIPALITY',
                'PROVINCE', 'MUNICIPALITY', 'ILLNESS1_DESC.1', 'ILLNESS2_DESC', 'ILLNESS_2_ACR_AMT',
                'ILLNESS_2_EST_AMT']

length_list = len(files_proc)

In [21]:
# actual amounts aggregation counts
for i in range(length_list):
    df_input = pd.read_csv(os.path.join(directory + files_proc[i] + '.csv'), low_memory=False)
    df_input.drop(cols_exclude, axis=1, inplace=True)
    df_input = df_input.loc[(df_input['YEAR'] != '2013') & (df_input['YEAR'] != '2014')]

    # computing for Turnaround Time
    df_input[['CHECK_DT','RECEIVED/REFILED DATE']] = df_input[['CHECK_DT','RECEIVED/REFILED DATE']].apply(pd.to_datetime)
    df_input['TAT'] = (df_input['CHECK_DT'] - df_input['RECEIVED/REFILED DATE']).dt.days

    # computing for Financial Coverage; set x/0 row as NaN so it will not be included in the aggregates
    df_input['FINANCIAL_COVERAGE'] = df_input['PAID AMOUNT']/df_input['ACTUAL_AMT']
    df_input.loc[~np.isfinite(df_input['FINANCIAL_COVERAGE']), 'FINANCIAL_COVERAGE'] = np.nan

    df_input.to_csv(os.path.join(directory, files_proc[i] + '_strip1.csv'), index=False)
    print(files_proc[i],len(df_input))
    
    del[df_input]
#     garbage_collector() 
    gc.collect()
    gc.collect()
    gc.collect()

90935 11988992
